In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

In [ ]:
file_path = './data/loan-predication.csv'
df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
print("Missing values before preprocessing:\n", df.isnull().sum())

cat_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Loan_Amount_Term', 'Credit_History']
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

num_cols = ['LoanAmount']
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

print("\nMissing values after preprocessing:\n", df.isnull().sum())

le = LabelEncoder()
categorical_to_encode = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

for col in categorical_to_encode:
    df[col] = le.fit_transform(df[col])

df.drop('Loan_ID', axis=1, inplace=True)

df.head()

In [ ]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

In [ ]:
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

y_pred = dt_classifier.predict(X_test)

In [ ]:
# Shallow Tree
shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
shallow_tree.fit(X_train, y_train)
y_pred_shallow = shallow_tree.predict(X_test)

# Deep Tree (Default)
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
deep_tree.fit(X_train, y_train)
y_pred_deep = deep_tree.predict(X_test)

print("Shallow Tree Accuracy:", accuracy_score(y_test, y_pred_shallow))
print("Deep Tree Accuracy:", accuracy_score(y_test, y_pred_deep))

In [ ]:
sample_data = X_test.iloc[0:5]
predictions = shallow_tree.predict(sample_data)
print("Predictions for first 5 test samples:", predictions)
print("Actual labels:", y_test.iloc[0:5].values)

In [ ]:
def evaluate_model(y_true, y_pred, tree_type):
    print(f"--- {tree_type} Evaluation ---")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall   :", recall_score(y_true, y_pred))
    print("F1-Score :", f1_score(y_true, y_pred))
    print("\n")

evaluate_model(y_test, y_pred_shallow, "Shallow Tree (max_depth=3)")
evaluate_model(y_test, y_pred_deep, "Deep Tree (Unpruned)")

In [ ]:
importances = shallow_tree.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print("Feature Importances:")
print(feature_importance_df)

In [ ]:
train_acc_deep = accuracy_score(y_train, deep_tree.predict(X_train))
test_acc_deep = accuracy_score(y_test, deep_tree.predict(X_test))

train_acc_shallow = accuracy_score(y_train, shallow_tree.predict(X_train))
test_acc_shallow = accuracy_score(y_test, shallow_tree.predict(X_test))

print(f"Deep Tree - Train Accuracy: {train_acc_deep:.4f}, Test Accuracy: {test_acc_deep:.4f}")
print(f"Shallow Tree - Train Accuracy: {train_acc_shallow:.4f}, Test Accuracy: {test_acc_shallow:.4f}")

if train_acc_deep - test_acc_deep > 0.1:
    print("\nSignificant overfitting detected in Deep Tree (Training accuracy is much higher than test accuracy).")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y_test, y_pred_shallow), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Shallow Tree)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_test, y_pred_deep), annot=True, fmt='d', cmap='Reds')
plt.title('Confusion Matrix (Deep Tree)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.tight_layout()
plt.show()

plt.figure(figsize=(20, 10))
plot_tree(shallow_tree, feature_names=X.columns.tolist(), class_names=['Rejected', 'Approved'], filled=True, rounded=True)
plt.title("Decision Tree Structure (Max Depth = 3)")
plt.show()

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance (Shallow Tree)')
plt.show()